# Price Elasticity

Estimate meal-level own-price elasticity with a controlled log-log model.

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [31]:
data = pd.read_csv(r'C:\Users\adaml\Documents\Stage\Pricing engine\Data\processed\data.csv')  # Load your data into a DataFrame

In [32]:
data.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,...,category_Other Snacks,category_Pasta,category_Pizza,category_Rice Bowl,category_Salad,category_Sandwich,category_Seafood,category_Soup,category_Starters,revenue
0,1,55,1885,136.83,152.29,0,0,177,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24218.91
1,1,55,1993,136.83,135.83,0,0,270,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36944.10
2,1,55,2539,134.86,135.86,0,0,189,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25488.54
3,1,55,2139,339.50,437.53,0,0,54,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18333.00
4,1,55,2631,243.50,242.50,0,0,40,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9740.00


In [33]:
weekly_meals = (
    data
    .groupby(['meal_id', 'week'], as_index=False)
    .agg({
        'num_orders': 'sum',
        'checkout_price': 'mean'
    })
)

In [34]:
weekly_meals.head()

,meal_id,week,num_orders,checkout_price
0,1062,1,28943,176.202727
1,1062,2,26695,180.710649
2,1062,3,27844,180.499221
3,1062,4,31144,180.664156
4,1062,5,29324,175.263816


In [35]:
weekly_meals = weekly_meals.sort_values(['meal_id', 'week'])

In [36]:
weekly_meals['previous_orders'] = weekly_meals.groupby('meal_id')['num_orders'].shift(1)

weekly_meals['previous_price'] = weekly_meals.groupby('meal_id')['checkout_price'].shift(1)

In [37]:
weekly_meals.head()

,meal_id,week,num_orders,checkout_price,previous_orders,previous_price
0,1062,1,28943,176.202727,NaN,NaN
1,1062,2,26695,180.710649,28943.0,176.202727
2,1062,3,27844,180.499221,26695.0,180.710649
3,1062,4,31144,180.664156,27844.0,180.499221
4,1062,5,29324,175.263816,31144.0,180.664156


In [38]:
weekly_meals['diff_orders'] = (
    weekly_meals['num_orders'] - weekly_meals['previous_orders']
)

weekly_meals['diff_price'] = (
    weekly_meals['checkout_price'] - weekly_meals['previous_price']
)

In [39]:
weekly_meals['orders_price_ratio'] = (
    weekly_meals['diff_orders'] / weekly_meals['diff_price']
)

In [40]:
import numpy as np

weekly_meals['orders_price_ratio'] = weekly_meals['orders_price_ratio'].replace(
    [np.inf, -np.inf],
    np.nan
)

In [41]:
avg_ratio_per_meal = (
    weekly_meals
    .groupby('meal_id', as_index=False)
    .agg(
        avg_orders_price_ratio=('orders_price_ratio', 'mean'),
        observations=('orders_price_ratio', 'count')
    )
)

In [42]:
avg_ratio_per_meal.head()

,meal_id,avg_orders_price_ratio,observations
0,1062,-9205.237034,144
1,1109,-1503.347371,144
2,1198,-152.501375,144
3,1207,-1973.915496,144
4,1216,4017.715838,143


In [43]:
weekly_meals['pct_change_orders'] = (
    weekly_meals.groupby('meal_id')['num_orders'].pct_change()
)

weekly_meals['pct_change_price'] = (
    weekly_meals.groupby('meal_id')['checkout_price'].pct_change()
)

In [44]:
weekly_meals['elasticity'] = (
    weekly_meals['pct_change_orders'] / weekly_meals['pct_change_price']
)

In [45]:
weekly_meals['elasticity'] = weekly_meals['elasticity'].replace(
    [np.inf, -np.inf],
    np.nan
)

In [46]:
avg_elasticity_per_meal = (
    weekly_meals
    .groupby('meal_id', as_index=False)
    .agg(
        avg_elasticity=('elasticity', 'mean'),
        median_elasticity=('elasticity', 'median'),
        observations=('elasticity', 'count')
    )
)

avg_elasticity_per_meal.head()

,meal_id,avg_elasticity,median_elasticity,observations
0,1062,-34.560752,-1.494312,144
1,1109,-8.724349,-1.298453,144
2,1198,-4.297801,-0.238326,144
3,1207,-37.672884,-1.421276,144
4,1216,464.082367,-1.267532,143


In [47]:
avg_elasticity_per_meal.to_csv(r'C:\Users\adaml\Documents\Stage\Pricing engine\Data\processed\avg_elasticity_per_meal.csv', index=False)